# Практика 3 — Преобразование и JOIN — профиль организаций HMDA

## Подготовка

In [ ]:
import json
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

## 01. Входы, строка и ключ

In [ ]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei":"string"})
assert len(organizations) == 470
assert not organizations.duplicated(org_keys).any()
assert not organizations[org_keys].isna().any().any()
assert not decisions[org_keys].isna().any().any()
organizations.head(4)

## 01. Входы, строка и ключ — шаг 2

In [ ]:
long = decisions.groupby(long_keys, dropna=False).size().rename("n").reset_index()
print(long.shape, long["n"].sum())
assert not long.duplicated(long_keys).any()
assert long["n"].sum() == len(decisions)
long.head(6)

## A. Завершите разворот

## 02. Обоснованный ноль и явные колонки

In [ ]:
wide = long.pivot(index=org_keys, columns="action_taken", values="n")
wide = wide.fillna(0).astype("int64")
wide = wide.rename(columns={1:"issued", 2:"approved_not_accepted", 3:"denied"})
wide = wide.rename_axis(None, axis=1).reset_index()
value_cols = ["issued", "approved_not_accepted", "denied"]
assert wide[value_cols].sum().sum() == 6830
wide.head(4)

## 02. Обоснованный ноль и явные колонки — шаг 2

In [ ]:
restored = wide.melt(id_vars=org_keys, value_vars=value_cols, var_name="outcome", value_name="n")
print(restored.shape)
restored.head(6)

## B. Восстановите наблюдаемые комбинации

## 03. Добавляем имя с контрактом

In [ ]:
lookup = organizations[org_keys + ["organization_name"]].copy()
enriched = decisions.merge(lookup, on=org_keys, how="left", validate="many_to_one", indicator=True)
print(enriched.shape)
print(enriched["_merge"].value_counts())
assert len(enriched) == 6830
assert enriched["source_row"].is_unique
assert set(enriched["source_row"]) == set(decisions["source_row"])
assert enriched["_merge"].eq("both").all()
enriched[["source_row", "lei", "organization_name", "action_taken"]].head(5)

## C. Проверьте сохранность самих значений

## 04. Неполная доставка: потеря при inner

In [ ]:
partial_lookup = lookup.iloc[1:].copy()
left_partial = decisions.merge(partial_lookup, on=org_keys, how="left", validate="many_to_one", indicator=True)
inner_partial = decisions.merge(partial_lookup, on=org_keys, how="inner", validate="many_to_one")
print(len(left_partial), len(inner_partial))
print(left_partial["_merge"].value_counts())
assert len(left_partial) == 6830
assert len(inner_partial) == 6805

## 04. Неполная доставка: потеря при inner — шаг 2

In [ ]:
replayed_lookup = pd.concat([lookup, lookup.iloc[[0]]], ignore_index=True)
wrong_join = decisions.merge(replayed_lookup, on=org_keys, how="left")
print("Без validate:", len(wrong_join))
assert len(wrong_join) == 6855
try:
    decisions.merge(replayed_lookup, on=org_keys, how="left", validate="many_to_one")
except pd.errors.MergeError:
    print("Повторный правый ключ остановлен")
else:
    raise AssertionError("Ожидалась остановка JOIN")

## D. Сохраните аудит покрытия

## 05. Профиль с собственным знаменателем

In [ ]:
prepared = decisions.copy()
prepared["is_approved"] = prepared["action_taken"].isin([1,2])
profile = prepared.groupby(org_keys).agg(n_decisions=("action_taken","size"), n_approved=("is_approved","sum")).reset_index()
profile["approval_rate"] = profile["n_approved"] / profile["n_decisions"]
named_profile = profile.merge(lookup, on=org_keys, how="left", validate="one_to_one")
assert len(named_profile) == 314
assert named_profile["n_decisions"].sum() == 6830
assert named_profile["n_approved"].sum() == 6139
named_profile.sort_values("n_decisions", ascending=False).head(5)

## E. Передайте проверяемый результат